# 🎙️ Entraînement d'une Voix Piper (VITS) — Thierry Blanc (Narrateur)

Ce notebook entraîne (fine-tune) un modèle **Piper (`OHF-Voice/piper1-gpl`)** en français à partir de votre dataset existant **`dataset_narrateur.zip`** (celui préparé pour OmniVoice), puis l'exporte au format **`.onnx` + `.onnx.json`** prêt pour **NekoSpeak**.

### ⚡ Pourquoi Piper (VITS) ?
- **Vraie voix de l'acteur** : contrairement au clonage zero-shot (Pocket-TTS), le réseau apprend directement les harmoniques, le grain et le phrasé de Thierry Blanc.
- **Ultra léger (~60 Mo)** : contre 2,45 Go pour OmniVoice.
- **Instantané sur S25 Ultra (~0,15 à 0,3 s par phrase)** : génération en une seule passe directe.

### Avant de lancer
1. Dans Colab : **Exécution → Modifier le type d'exécution → GPU** (T4, L4 ou A100).
2. Assurez-vous que `dataset_narrateur.zip` est présent dans votre Google Drive (par défaut `Mon Drive/omnivoice/dataset_narrateur.zip`).
3. Les checkpoints et le modèle final seront sauvegardés automatiquement sur votre Drive (`Mon Drive/piper_narrateur/`).

## 1. Configuration & Montage Google Drive

In [ ]:
import os
import torch
from google.colab import drive

assert torch.cuda.is_available(), (
    "❌ Aucun GPU NVIDIA (CUDA) détecté ! "
    "Dans Colab : Exécution → Modifier le type d'exécution → GPU (T4, L4 ou A100). "
    "(Les TPU ne sont pas compatibles avec Piper/VITS.)"
)

if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# === PARAMÈTRES GÉNÉRAUX ===
VOICE_NAME = "fr_FR-narrateur_thierry-medium"
DATASET_ZIP = "/content/drive/MyDrive/omnivoice/dataset_narrateur.zip"
DRIVE_OUT_DIR = "/content/drive/MyDrive/piper_narrateur"

MAX_EPOCHS = 300       # 150-300 époques recommandées en fine-tuning sur ~2h d'audio
SAMPLE_RATE = 22050    # Standard Piper medium
AUTO_DISCONNECT = True   # Coupe automatiquement le GPU Colab à la fin de l'export (Cellule 6)

# === AUTO-CONFIGURATION SELON LE GPU DÉTECTÉ ===
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
cpu_cores = os.cpu_count() or 2
major_cc, _ = torch.cuda.get_device_capability(0)

if vram_gb >= 38:          # A100 (40/80 Go), H100
    BATCH_SIZE = 32
    PRECISION = "bf16"
    NUM_WORKERS = min(4, cpu_cores)
elif vram_gb >= 20:        # L4 (24 Go), A10G
    BATCH_SIZE = 24
    PRECISION = "bf16" if major_cc >= 8 else "32"
    NUM_WORKERS = min(4, cpu_cores)
elif vram_gb >= 14:        # T4 (15-16 Go), V100, P100
    BATCH_SIZE = 16
    PRECISION = "32"
    NUM_WORKERS = min(2, cpu_cores)
else:                      # GPU < 14 Go
    BATCH_SIZE = 8
    PRECISION = "32"
    NUM_WORKERS = min(2, cpu_cores)

print(f"🖥️ GPU détecté : {gpu_name} ({vram_gb:.1f} Go VRAM, Compute {major_cc}.x, {cpu_cores} CPU)")
print(f"⚙️ Config auto  : BATCH_SIZE={BATCH_SIZE} | PRECISION={PRECISION} | NUM_WORKERS={NUM_WORKERS}")

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
WORK_DIR = "/content/piper_work"
os.makedirs(WORK_DIR, exist_ok=True)

# Correctif PyTorch 2.6+ pour le chargement des checkpoints Lightning
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

assert os.path.exists(DATASET_ZIP), f"Dataset introuvable : {DATASET_ZIP}"
print(f"✅ Dataset trouvé : {DATASET_ZIP} ({os.path.getsize(DATASET_ZIP)/1e6:.1f} Mo)")


## 2. Installation de `OHF-Voice/piper1-gpl` et dépendances système
Installation des compilateurs nécessaires pour `monotonic_align` et `espeak-ng`.

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq build-essential cmake ninja-build espeak-ng ffmpeg

cd /content
if [ ! -d "piper1-gpl" ]; then
    git clone --depth 1 https://github.com/OHF-Voice/piper1-gpl.git
fi
cd /content/piper1-gpl
pip install -q -e ".[train]" huggingface_hub librosa soundfile onnxruntime scikit-build scikit-build-core
python3 setup.py build_ext --inplace
if [ -f "./build_monotonic_align.sh" ]; then
    chmod +x ./build_monotonic_align.sh && ./build_monotonic_align.sh
fi
python3 -c "import sys; sys.path.insert(0, "src"); from piper import espeakbridge; print("✅ espeakbridge OK !")"


## 3. Préparation du Dataset (`22050 Hz` mono + `metadata.csv`)
Extraction de `dataset_narrateur.zip`, ré-échantillonnage des clips audio à 22 050 Hz mono PCM 16-bit et création du fichier `metadata.csv` au format attendu par Piper (`fichier.wav|texte`).

In [ ]:
import json
import zipfile
import shutil
import soundfile as sf
import librosa
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

RAW_DATA_DIR = os.path.join(WORK_DIR, "raw_data")
PIPER_WAV_DIR = os.path.join(WORK_DIR, "wavs")
os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PIPER_WAV_DIR, exist_ok=True)

print("📦 Extraction de dataset_narrateur.zip...")
with zipfile.ZipFile(DATASET_ZIP, "r") as z:
    z.extractall(RAW_DATA_DIR)

# Recherche récursive des manifestes train.jsonl / dev.jsonl et des fichiers .wav
entries = []
wav_index = {}
for root, _, files in os.walk(RAW_DATA_DIR):
    for fn in files:
        full_path = os.path.join(root, fn)
        if fn in ("train.jsonl", "dev.jsonl"):
            with open(full_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        entries.append(json.loads(line))
        elif fn.lower().endswith(".wav"):
            wav_index[fn] = full_path

print(f"📄 {len(entries)} clips trouvés dans les manifestes ({len(wav_index)} fichiers .wav indexés).")
assert len(entries) > 0, f"Aucun manifeste train.jsonl/dev.jsonl trouvé dans {RAW_DATA_DIR}"

def process_clip(item):
    orig_path = item.get("audio_path", "")
    base_name = os.path.basename(orig_path)
    src = orig_path if os.path.exists(orig_path) else wav_index.get(base_name)
    if not src or not os.path.exists(src):
        return None
    text = item.get("text", "").replace("|", " ").replace("
", " ").strip()
    if not text:
        return None
    dst_name = f"{item.get('id', os.path.splitext(base_name)[0])}.wav"
    dst_path = os.path.join(PIPER_WAV_DIR, dst_name)
    if not os.path.exists(dst_path):
        y, sr = librosa.load(src, sr=SAMPLE_RATE, mono=True)
        dur = len(y) / SAMPLE_RATE
        if dur < 0.8 or dur > 15.0:
            return None
        sf.write(dst_path, y, SAMPLE_RATE, subtype="PCM_16")
    return f"{dst_name}|{text}"

csv_lines = []
with ThreadPoolExecutor(max_workers=4) as ex:
    for res in tqdm(ex.map(process_clip, entries), total=len(entries), desc="Conversion 22050Hz"):
        if res:
            csv_lines.append(res)

assert len(csv_lines) > 0, "Aucune ligne générée dans metadata.csv ! Vérifiez les chemins audio."
CSV_PATH = os.path.join(WORK_DIR, "metadata.csv")
with open(CSV_PATH, "w", encoding="utf-8") as f:
    f.write("
".join(csv_lines) + "
")

print(f"✅ Dataset Piper prêt : {len(csv_lines)} phrases -> {CSV_PATH}")


## 4. Téléchargement d'un Checkpoint Français de Base (`medium`)
On télécharge un checkpoint officiel français (`rhasspy/piper-checkpoints`, qualité `medium`, 22 050 Hz) pour démarrer l'entraînement par transfert (warmstart) : le modèle connaît déjà la phonétique française, il n'a plus qu'à apprendre la voix de Thierry Blanc !

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download

REPO_CKPT = "rhasspy/piper-checkpoints"
files = list_repo_files(REPO_CKPT, repo_type="dataset")

# On cherche en priorité fr/fr_FR/upmc/medium ou fr/fr_FR/siwis/medium ou n'importe quel fr_FR medium .ckpt
fr_ckpts = [f for f in files if f.startswith("fr/fr_FR/") and "/medium/" in f and f.endswith(".ckpt")]
print("Checkpoints français disponibles :", fr_ckpts)

preferred = [f for f in fr_ckpts if "siwis" in f or "upmc" in f or "tom" in f]
ckpt_rel_path = preferred[0] if preferred else fr_ckpts[0]

print(f"⬇️ Téléchargement du checkpoint de base : {ckpt_rel_path}...")
BASE_CKPT_PATH = hf_hub_download(
    repo_id=REPO_CKPT,
    filename=ckpt_rel_path,
    repo_type="dataset",
    local_dir=os.path.join(WORK_DIR, "base_ckpt")
)
print(f"✅ Checkpoint téléchargé : {BASE_CKPT_PATH}")

## 5. Lancement de l'Entraînement (Fine-Tuning)
Les checkpoints sont sauvegardés dans `Mon Drive/piper_narrateur/training_logs/` pour ne rien perdre si Colab se déconnecte.

In [ ]:
import glob, shutil, subprocess, threading, time

CACHE_DIR = os.path.join(WORK_DIR, "cache")
CONFIG_OUT_PATH = os.path.join(WORK_DIR, f"{VOICE_NAME}.onnx.json")
LOCAL_LOGS_DIR = os.path.join(WORK_DIR, "training_logs")
DRIVE_CKPT_DIR = os.path.join(DRIVE_OUT_DIR, "checkpoints_backup")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_LOGS_DIR, exist_ok=True)
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

# Vérifier s'il existe déjà un checkpoint en local ou sur Google Drive
local_ckpts = sorted(glob.glob(f"{LOCAL_LOGS_DIR}/lightning_logs/version_*/checkpoints/last.ckpt"))
drive_last = os.path.join(DRIVE_CKPT_DIR, "last.ckpt")
drive_old_ckpts = sorted(glob.glob(f"{DRIVE_OUT_DIR}/training_logs/lightning_logs/version_*/checkpoints/last.ckpt"))

resume_ckpt = None
if local_ckpts:
    resume_ckpt = local_ckpts[-1]
elif os.path.exists(drive_last):
    resume_ckpt = drive_last
elif drive_old_ckpts:
    # Copier l'ancien checkpoint Drive en local pour accélérer le chargement
    shutil.copy2(drive_old_ckpts[-1], drive_last)
    resume_ckpt = drive_last

cmd = [
    "python3", "-m", "piper.train", "fit",
    f"--data.voice_name={VOICE_NAME}",
    f"--data.csv_path={CSV_PATH}",
    f"--data.audio_dir={PIPER_WAV_DIR}",
    f"--model.sample_rate={SAMPLE_RATE}",
    "--data.espeak_voice=fr",
    f"--data.cache_dir={CACHE_DIR}",
    f"--data.config_path={CONFIG_OUT_PATH}",
    f"--data.batch_size={BATCH_SIZE}",
    f"--data.num_workers={NUM_WORKERS}",
    "--data.validation_split=0.05",
    "--data.num_test_examples=2",
    f"--trainer.max_epochs={MAX_EPOCHS}",
    "--trainer.accelerator=gpu",
    "--trainer.devices=1",
    f"--trainer.precision={PRECISION}",
    f"--trainer.default_root_dir={LOCAL_LOGS_DIR}"
]

if resume_ckpt:
    print(f"🔄 Reprise depuis : {resume_ckpt}")
    cmd.append(f"--ckpt_path={resume_ckpt}")
else:
    print(f"🚀 Nouvel entraînement (warmstart) depuis : {BASE_CKPT_PATH}")
    cmd.append(f"--model.warmstart_ckpt={BASE_CKPT_PATH}")

# Sauvegarde sur Google Drive uniquement toutes les 10 minutes (évite de saturer Drive à chaque époque)
stop_sync = threading.Event()
def sync_to_drive():
    while not stop_sync.wait(600):  # toutes les 10 min
        ckpts = sorted(glob.glob(f"{LOCAL_LOGS_DIR}/lightning_logs/version_*/checkpoints/last.ckpt"))
        if ckpts:
            try:
                shutil.copy2(ckpts[-1], drive_last)
                print("
☁️ [Backup Drive] last.ckpt sauvegardé sur Google Drive.")
            except Exception as e:
                print(f"
⚠️ [Backup Drive] Erreur : {e}")

sync_thread = threading.Thread(target=sync_to_drive, daemon=True)
sync_thread.start()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
process = subprocess.Popen(
    cmd,
    cwd="/content/piper1-gpl",
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)
try:
    for line in process.stdout:
        print(line, end="")
    ret = process.wait()
finally:
    stop_sync.set()
    # Backup final à l'arrêt ou en fin d'entraînement
    ckpts = sorted(glob.glob(f"{LOCAL_LOGS_DIR}/lightning_logs/version_*/checkpoints/last.ckpt"))
    if ckpts:
        shutil.copy2(ckpts[-1], drive_last)
        print("
☁️ [Backup Drive final] last.ckpt sauvegardé sur Google Drive.")

if ret != 0:
    raise RuntimeError(f"L'entraînement s'est terminé avec le code {ret}")


## 6. Export ONNX et Test d'Écoute
On exporte le meilleur checkpoint (ou `last.ckpt`) vers `.onnx` et `.onnx.json`, puis on génère un extrait audio de test en français.

In [ ]:
import glob, shutil, subprocess
import IPython.display as ipd

local_ckpts = sorted(glob.glob(f"{WORK_DIR}/training_logs/lightning_logs/version_*/checkpoints/*.ckpt"))
drive_backup = os.path.join(DRIVE_OUT_DIR, "checkpoints_backup", "last.ckpt")
drive_old = sorted(glob.glob(f"{DRIVE_OUT_DIR}/training_logs/lightning_logs/version_*/checkpoints/*.ckpt"))

ckpts = local_ckpts if local_ckpts else ([drive_backup] if os.path.exists(drive_backup) else drive_old)
assert ckpts, "Aucun checkpoint trouvé !"
last_ckpts = [c for c in ckpts if c.endswith("last.ckpt")]
best_ckpt = last_ckpts[-1] if last_ckpts else ckpts[-1]
print(f"📦 Export de : {best_ckpt}")

ONNX_OUT = os.path.join(DRIVE_OUT_DIR, f"{VOICE_NAME}.onnx")
JSON_OUT = os.path.join(DRIVE_OUT_DIR, f"{VOICE_NAME}.onnx.json")

subprocess.run([
    "python3", "-m", "piper.train.export_onnx",
    "--checkpoint", best_ckpt,
    "--output-file", ONNX_OUT
], cwd="/content/piper1-gpl", check=True)

shutil.copy(CONFIG_OUT_PATH, JSON_OUT)
print(f"✅ Modèle exporté : {ONNX_OUT} ({os.path.getsize(ONNX_OUT)/1e6:.1f} Mo)")
print(f"✅ Config exportée : {JSON_OUT}")

# Test d'écoute avec Piper CLI
TEST_TEXT = "Il déambula dans les rues désertes et les jardins publics d'Agrillon, réfléchissant avec gravité aux propos de l'estropier et le maudissant."
TEST_WAV = os.path.join(WORK_DIR, "test_piper.wav")

subprocess.run([
    "python3", "-m", "piper",
    "-m", ONNX_OUT,
    "-c", JSON_OUT,
    "-f", TEST_WAV
], input=TEST_TEXT, text=True, check=True)

ipd.display(ipd.Audio(TEST_WAV))

# Extinction automatique du runtime Colab pour économiser les crédits
if globals().get("AUTO_DISCONNECT", False):
    import time
    from google.colab import runtime
    print("⏳ Attente 10s pour finaliser la synchronisation Google Drive...")
    time.sleep(10)
    print("🔌 Extinction automatique du runtime Colab (crédits préservés).")
    runtime.unassign()


## 7. Intégration dans NekoSpeak

Une fois les deux fichiers générés dans votre Drive (`Mon Drive/piper_narrateur/`) :
- `fr_FR-narrateur_thierry-medium.onnx` (~60 Mo)
- `fr_FR-narrateur_thierry-medium.onnx.json` (~5 Ko)

Vous pouvez les déposer sur votre dépôt Hugging Face et les déclarer dans `VoiceRepository.kt` (`loadCustomVoices()`), ou les copier directement dans le stockage interne de NekoSpeak sous `files/piper/fr_FR-narrateur_thierry-medium/` !